> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 5 — Research Board: Shared Agent State (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2005.%20Building%20Personal%20Assistants/LC4LSH_Chapter_5_Research_Board.ipynb)

**Learning objectives**
- Model shared state for hypotheses, evidence, and contradictions
- Let agents post and update items on a common board
- Track tasks and provenance
- Resolve contradictions explicitly

> Runtime: ~10 min (API)  
> Cost: paid LLM required  
> Data: synthetic research items


## Environment setup


### Secrets (Colab or local)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter5-research-board"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## What is a research board?

A **research board** is shared, structured state that multiple agents (or one agent over many turns) read and write. It holds:

- **Hypotheses** — candidate explanations
- **Evidence** — supporting observations with provenance
- **Contradictions** — conflicting findings to resolve
- **Tasks** — open questions / next steps

This mirrors how a lab notebook or Kanban board coordinates a research team.


## 1. Define the board schema


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import time, uuid

class Item(BaseModel):
    id: str = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    kind: Literal["hypothesis", "evidence", "contradiction", "task"]
    text: str
    source: str = "agent"
    ts: float = Field(default_factory=time.time)

class Board(BaseModel):
    items: list[Item] = []

    def post(self, kind, text, source="agent"):
        it = Item(kind=kind, text=text, source=source)
        self.items.append(it)
        return it

    def by_kind(self, kind):
        return [i for i in self.items if i.kind == kind]

board = Board()
print("Board ready")


## 2. Populate the board


In [ ]:
board.post("hypothesis", "Compound X inhibits kinase Y", source="planner")
board.post("evidence", "In-vitro assay shows 70% inhibition at 1uM", source="lab-agent")
board.post("evidence", "Docking score -8.2 kcal/mol", source="sim-agent")
board.post("contradiction", "Cell assay shows no phenotype change", source="lab-agent")
board.post("task", "Repeat cell assay with longer exposure", source="planner")

for it in board.items:
    print(f"[{it.kind:13}] ({it.source}) {it.text}")


## 3. Agents that read/write the board


In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def board_summary(board):
    return "
".join(f"[{i.kind}] {i.text}" for i in board.items)

def planner_agent(board):
    """Read the board and propose a next task given open contradictions."""
    contradictions = board.by_kind("contradiction")
    prompt = (f"Research board:
{board_summary(board)}

"
              f"There are {len(contradictions)} open contradictions. "
              f"Propose ONE concrete next task. Reply with just the task.")
    task = llm.invoke(prompt).content.strip()
    board.post("task", task, source="planner")
    return task

new_task = planner_agent(board)
print("Planner added task:", new_task)


## 4. Resolve a contradiction


In [ ]:
def resolve_contradiction(board, contradiction_text, resolution):
    """Mark a contradiction as resolved by posting follow-up evidence."""
    board.post("evidence", f"RESOLUTION of '{contradiction_text}': {resolution}", source="resolver")

resolve_contradiction(board, "Cell assay shows no phenotype change",
                      "Assay repeated at 48h shows expected phenotype; initial exposure too short.")

print("Evidence now:")
for e in board.by_kind("evidence"):
    print(" -", e.text)


## 5. Export the board


In [ ]:
import json
print(board.model_dump_json(indent=2)[:600], "...")


## Limitations & safety notes

- The board is in-memory only; persist it (DB/file) for real multi-agent systems.
- No access control — any agent can overwrite; add versioning for production.
- The planner is a single LLM call, not a full planning loop.
- **Paid API required**.


In [ ]:
# Cleanup
import gc
for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why shared structured state over message passing?</summary>It gives all agents a consistent, queryable view and avoids lost context in long conversations.</details>

<details><summary>What is provenance here?</summary>Each item records its source agent, so you can trace where a claim came from.</details>

<details><summary>Why model contradictions explicitly?</summary>Surfacing conflicts lets the system schedule resolution work instead of silently ignoring inconsistent results.</details>

### Tasks
- **Task A** - Add a `status` field (open/resolved) to contradictions and filter resolved ones.
- **Task B** - Add a critic agent that flags weak evidence (no source) for review.
- **Task C** - Persist the board to disk and reload it between runs.
- **Task D** - Add a voting mechanism so multiple agents can up/down-vote a hypothesis.
